# Сбор данных

In [1]:
%%capture
!pip install requests, bs4, urljoin, time

In [2]:
import time
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

In [3]:
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/123.0.0.0 Safari/537.36"
    )
}

In [4]:
def load_page(url, retries=3, pause=5):
    for attempt in range(retries):
        try:
            response = requests.get(url, headers=HEADERS, timeout=30)
            response.raise_for_status()
            return BeautifulSoup(response.text, "html.parser")

        except requests.exceptions.RequestException as e:
            print(f"Ошибка при загрузке страницы: {e}")

            if attempt < retries - 1:
                print(f"Повторная попытка через {pause} секунд...")
                time.sleep(pause)
            else:
                print("Не удалось загрузить страницу.")
                return None

In [5]:
def extract_news(article):
    news_id = article.get("id", "-")

    title_tag = article.find("a", attrs={"data-test-id": "article-snippet-title-link"})
    title = title_tag.get_text(strip=True) if title_tag else "-"
    link = urljoin("https://habr.com", title_tag.get("href")) if title_tag else "-"

    author_tag = article.find("a", class_="tm-user-info__username")
    author = author_tag.get_text(strip=True) if author_tag else "-"

    complexity = "-"
    for text in article.stripped_strings:
        if text in ["Простой", "Средний", "Сложный"]:
            complexity = text
            break

    return {
        "author": author,
        "complexity": complexity,
        "id": news_id,
        "link": link,
        "title": title
    }

In [6]:
def extract_next_page(page):
    next_page_tag = page.find("a", rel="next")

    if next_page_tag and next_page_tag.get("href"):
        return urljoin("https://habr.com", next_page_tag.get("href"))

    return None

In [7]:
def get_news(url, n_pages=1):
    news_list = []
    current_url = url

    for _ in range(n_pages):
        if current_url is None:
            break

        print(f"Collecting data from page: {current_url}")

        page = load_page(current_url)

        if page is None:
            break

        articles = page.find_all("article", attrs={"data-test-id": "articles-list-item"})
        if not articles:
            articles = [article for article in page.find_all("article") if article.get("id")]

        for article in articles:
            news_list.append(extract_news(article))

        current_url = extract_next_page(page)

    return news_list

In [16]:
news_list = get_news("https://habr.com/ru/articles/", n_pages=15)
news_list[:1]

[{'author': 'tr0llcr4ck',
  'complexity': 'Средний',
  'id': '1020334',
  'link': 'https://habr.com/ru/articles/1020334/',
  'title': 'Сборка локального вычислительного кластера на двух процессорах и майнинговых видеокартах'}]

# Сохранение данных в БД

In [17]:
%%capture
!pip install sqlalchemy

In [18]:
from sqlalchemy.orm import declarative_base
from sqlalchemy import Column, String, Integer
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker

In [19]:
Base = declarative_base()
engine = create_engine("sqlite:///news.db")
Session = sessionmaker(bind=engine)

In [20]:
class News(Base):
    __tablename__ = "news"

    id = Column(Integer, primary_key=True)
    title = Column(String)
    author = Column(String)
    url = Column(String)
    complexity = Column(String)
    habr_id = Column(String)
    label = Column(String)

In [21]:
Base.metadata.create_all(bind=engine)

In [22]:
s = Session()

In [23]:
for item in news_list:
    news = News(
        title=item["title"],
        author=item["author"],
        url=item["link"],
        complexity=item["complexity"],
        habr_id=item["id"],
        label=None
    )
    s.add(news)

s.commit()

In [24]:
all_news = s.query(News).all()

for news in all_news[:5]:
    print(news.id, news.title, news.author, news.habr_id)

1 От промпта к мутациям: как я перестал писать тесты руками и собрал команду из 7 AI-агентов M0SEY 1020066
2 Как выбрать гидроцилиндр? boris_green 1020146
3 Объясняю на пальцах — зачем твоему бизнесу каталог данных veloriant 1020142
4 Какой я аналитик? Andrianov_Vladislav 1020014
5 «Невидимая» революция инвестиционных приоритетов в США boris_green 1020140


# Разметка данных

In [25]:
%%capture
!pip install bottle

In [26]:
from bottle import route, template, run, request, redirect
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy import Column, String, Integer, create_engine
from sqlalchemy.orm import sessionmaker

In [31]:
from sqlalchemy import create_engine
from sqlalchemy.pool import NullPool

engine = create_engine(
    "sqlite:///news.db",
    poolclass=NullPool,
    connect_args={"check_same_thread": False}
)

In [32]:
@route('/news')
def news_list():
    s = Session()
    rows = s.query(News).filter(News.label == None).all()
    s.close()
    return template('news_template', rows=rows)

In [33]:
@route('/add_label')
def add_label():
    # 1. Получить значения параметров
    label = request.query.get('label')
    news_id = request.query.get('id')

    # 2. Найти запись
    s = Session()
    row = s.query(News).filter(News.id == news_id).first()

    # 3–4. Обновить и сохранить
    if row is not None:
        row.label = label
        s.commit()

        s.close()

    # ВАЖНО: внутри функции
    if __name__ == "__main__":
        redirect('/news')

In [34]:
run(host='localhost', port=8080, debug=True, reloader=False)

Bottle v0.13.4 server starting up (using WSGIRefServer())...
Listening on http://localhost:8080/
Hit Ctrl-C to quit.

127.0.0.1 - - [07/Apr/2026 15:05:23] "GET /news HTTP/1.1" 200 348154
127.0.0.1 - - [07/Apr/2026 15:05:35] "GET /add_label?label=never&id=47 HTTP/1.1" 303 0
127.0.0.1 - - [07/Apr/2026 15:05:35] "GET /news HTTP/1.1" 200 347491
127.0.0.1 - - [07/Apr/2026 15:05:40] "GET /add_label?label=never&id=52 HTTP/1.1" 303 0
127.0.0.1 - - [07/Apr/2026 15:05:40] "GET /news HTTP/1.1" 200 346898
127.0.0.1 - - [07/Apr/2026 15:06:02] "GET /add_label?label=maybe&id=56 HTTP/1.1" 303 0
127.0.0.1 - - [07/Apr/2026 15:06:02] "GET /news HTTP/1.1" 200 345615
127.0.0.1 - - [07/Apr/2026 15:06:06] "GET /add_label?label=never&id=75 HTTP/1.1" 303 0
127.0.0.1 - - [07/Apr/2026 15:06:06] "GET /news HTTP/1.1" 200 344963
127.0.0.1 - - [07/Apr/2026 15:06:11] "GET /add_label?label=maybe&id=83 HTTP/1.1" 303 0
127.0.0.1 - - [07/Apr/2026 15:06:11] "GET /news HTTP/1.1" 200 344313
127.0.0.1 - - [07/Apr/2026 15:06:

In [35]:
import os
print(os.getcwd())
print(os.listdir())

/Users/arinakonoplina/PycharmProjects/JupyterProject
['news.db', 'news_template.tpl', 'models', 'README.md', 'sample.ipynb', '.venv', 'Lab5_PDA', 'news.db-journal', '.idea']
